# <font color='black'>Регрессионный анализ: продолжение </font>
## <font color='black'> Практическое занятие 5. Гребневая регрессия </font>

Краткое описание данных:

* lngdp2 - логарифм ВВП на душу населения - зависимая переменная

В качестве предикторов выступают 6 индексов качества гос. управления WGI (Worldwide Governance Indicators)
* va - voice and accountability
* rl - rule and law
* rq - regulatory quality
* gove - government effectiveness
* ps - political stability
* cc - control of corruption

Распределения этих индексов приведено в рамках проекта к стандартному нормальному, поэтому мы не используем дополнительно стандартизацию

In [58]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [59]:
dta = pd.read_stata('data_ridge.dta')
dta = dta.dropna()
dta.head()

,lngdp2,va,rl,rq,gove,ps,cc,gdp2,country
0,5.662960,-1.40,-1.76,-1.35,-1.80,-2.45,-1.31,288.0,43.0
1,7.765145,-1.06,-0.95,-0.56,-0.60,-2.39,-0.82,2357.0,140.0
2,7.538495,-1.68,-1.47,-1.25,-1.15,-2.29,-1.26,1879.0,176.0
3,9.210740,-0.45,-0.74,-0.21,-0.59,-2.22,-0.69,10004.0,99.0
4,6.956545,-1.26,-1.46,-1.14,-1.43,-2.12,-1.34,1050.0,75.0


Оценим модель m1 на исходных данных. Можем ли мы доверять полученным результатам?

In [60]:
m1 = smf.ols(formula = "lngdp2 ~ va + rl + rq + gove + ps + cc", data = dta).fit()
print(m1.summary())

                            OLS Regression Results                            
Dep. Variable:                 lngdp2   R-squared:                       0.657
Model:                            OLS   Adj. R-squared:                  0.645
Method:                 Least Squares   F-statistic:                     51.81
Date:                Tue, 20 Oct 2026   Prob (F-statistic):           3.09e-35
Time:                        08:30:14   Log-Likelihood:                -191.70
No. Observations:                 169   AIC:                             397.4
Df Residuals:                     162   BIC:                             419.3
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      8.6997      0.060    145.828      0.0

Выведем корреляционную матрицу для наших предикторов:

In [61]:
dta[["va", "rl", "rq", "gove", "ps", "cc"]].corr().round(3)

,va,rl,rq,gove,ps,cc
va,1.000,0.774,0.788,0.751,0.651,0.762
rl,0.774,1.000,0.890,0.943,0.770,0.952
rq,0.788,0.890,1.000,0.943,0.647,0.877
gove,0.751,0.943,0.943,1.000,0.687,0.940
ps,0.651,0.770,0.647,0.687,1.000,0.711
cc,0.762,0.952,0.877,0.940,0.711,1.000


Кроме этого, понять, есть ли у нас свидетельства в пользу сильной мультиколлинеарности, нам помогут коэффициенты VIF (коэффициенты "вздутия" дисперсии). Проинтерпретируйте полученные результаты

In [62]:
X = dta[["va", "rl", "rq", "gove", "ps", "cc"]]
X = add_constant(X)

vif_data = pd.DataFrame({'variables':X.columns[1:], 'VIF':[variance_inflation_factor(X.values, i+1) for i in range(len(X.columns[1:]))]})
print(vif_data)

  variables        VIF
0        va   3.099573
1        rl  17.089986
2        rq  10.764995
3      gove  20.736053
4        ps   2.629430
5        cc  13.327886


 Применим гребневую регрессию для получения более устойчивых результатов. Воспользуемся процедурой кросс-валидации для подбора оптимального параметра регуляризации. Разделим наш массив на 2 подвыборки: тестовую (20% данных) и обучающую (соответственно, 80% данных)

In [63]:
train, test = train_test_split(dta, test_size = 0.2, random_state = 11)

Для начала используем 1 в качестве параметра регуляризации и выведем оценки коэффициентов в модели гребневой регрессии:

In [64]:
ridge1 = Ridge(alpha = 1)
ridge1.fit(train[["va", "rl", "rq", "gove", "ps", "cc"]], train["lngdp2"])
ridge1.coef_

array([-0.07996015, -0.07944857,  0.21281882,  0.96958308,  0.24193523,
       -0.16828073])

Напишем функцию для расчета стандартных ошибок:

In [65]:
def ridge_se(model, X, y):
    X = np.asarray(X); y = np.asarray(y).ravel()
    n, p = X.shape
    ols = LinearRegression().fit(X, y)
    sigma_sq = np.sum((y - ols.predict(X))**2) / (n - p - 1)
    XTX = X.T @ X
    A = XTX + model.alpha * np.eye(p)
    B = np.linalg.solve(A, XTX)
    return np.sqrt(np.diag(sigma_sq * np.linalg.solve(A, B)))

Рассчитаем стандартные ошибки для оценок коэффициентов нашей первой модели гребневой регрессии:

In [66]:
X = train[["va", "rl", "rq", "gove", "ps", "cc"]]
y = train['lngdp2']
se1 = ridge_se(ridge1, X, y)

se1

array([0.1265961 , 0.23515803, 0.19544249, 0.24781456, 0.11724244,
       0.2097062 ])

Для удобства представим ниже таблицу, в которой сравним оценки коэффициентов и их значимость в исходной модели и модели гребневой регрессии (с параметром $\alpha$ = 1):

In [67]:
ridge1_data = pd.DataFrame({'variables':X.columns, 'coef': m1.params[1:], 'se': m1.bse[1:], 't': m1.params[1:]/m1.bse[1:],
                            'coef_ridge':ridge1.coef_, 'se_ridge': se1, 't_ridge': ridge1.coef_/se1})
print(ridge1_data)

     variables      coef        se         t  coef_ridge  se_ridge   t_ridge
va          va -0.175811  0.108381 -1.622160   -0.079960  0.126596 -0.631616
rl          rl -0.064419  0.253270 -0.254350   -0.079449  0.235158 -0.337852
rq          rq  0.300197  0.205736  1.459135    0.212819  0.195442  1.088908
gove      gove  1.067160  0.274419  3.888790    0.969583  0.247815  3.912535
ps          ps  0.265687  0.103624  2.563944    0.241935  0.117242  2.063547
cc          cc -0.296014  0.218021 -1.357736   -0.168281  0.209706 -0.802459


Подберем оптимальный параметр регуляризации. Для этого используем k-блочную кросс-валидацию: то есть, массив разбивается на k равных подвыборок, далее проводим для каждого заданного параметра $\alpha$ k итераций: j-ая подвыборка выступает тестовой, остальные подвыборки составляют обучающую. Считаем среднее MSE по k итерациям для каждого значения $\alpha$ и далее останавливаемся на том значении $\alpha$, при котором усредненное MSE принимает минимальное значение

In [68]:
alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1000]

grid_search = GridSearchCV(ridge1, {'alpha': alphas}, cv = 5)
grid_search.fit(train[["va", "rl", "rq", "gove", "ps", "cc"]], train["lngdp2"])

print("Best Regularization Parameter:", grid_search.best_params_)

Best Regularization Parameter: {'alpha': 10}


In [69]:
ridge2 = Ridge(alpha = 10)
ridge2.fit(train[["va", "rl", "rq", "gove", "ps", "cc"]], train["lngdp2"])
ridge2.coef_

array([-0.05072072,  0.0787482 ,  0.29460551,  0.54860055,  0.18126691,
        0.02137558])

In [70]:
se2 = ridge_se(ridge2, X, y)

se2

array([0.09676798, 0.10702097, 0.1060998 , 0.10151109, 0.09102234,
       0.10627653])

In [71]:
ridge2_data = pd.DataFrame({'variables':X.columns, 'coef': m1.params[1:], 'se': m1.bse[1:], 't': m1.params[1:]/m1.bse[1:],
                            'coef_ridge':ridge2.coef_, 'se_ridge': se2, 't_ridge': ridge2.coef_/se2})
print(ridge2_data)

     variables      coef        se         t  coef_ridge  se_ridge   t_ridge
va          va -0.175811  0.108381 -1.622160   -0.050721  0.096768 -0.524148
rl          rl -0.064419  0.253270 -0.254350    0.078748  0.107021  0.735820
rq          rq  0.300197  0.205736  1.459135    0.294606  0.106100  2.776683
gove      gove  1.067160  0.274419  3.888790    0.548601  0.101511  5.404341
ps          ps  0.265687  0.103624  2.563944    0.181267  0.091022  1.991455
cc          cc -0.296014  0.218021 -1.357736    0.021376  0.106277  0.201132


$R^2$ и $MSE$ на тестовой и обучающей подвыборках:

In [72]:
y_pred_test = ridge2.predict(test[["va", "rl", "rq", "gove", "ps", "cc"]])
mse_test = mean_squared_error(test["lngdp2"], y_pred_test)

In [73]:
r2_test = r2_score(test["lngdp2"], y_pred_test)

In [74]:
y_pred_train = ridge2.predict(train[["va", "rl", "rq", "gove", "ps", "cc"]])
mse_train = mean_squared_error(train["lngdp2"], y_pred_train)

In [75]:
r2_train = r2_score(train["lngdp2"], y_pred_train)

In [76]:
print(f'R2 обучающая выборка: {r2_train:.3f}\nR2 тестовая выборка: {r2_test:.3f}')
print(f'MSE обучающая выборка: {mse_train:.3f}\nMSE тестовая выборка: {mse_test:.3f}')

R2 обучающая выборка: 0.640
R2 тестовая выборка: 0.583
MSE обучающая выборка: 0.598
MSE тестовая выборка: 0.605
